# 04b - Pose Visualization
Overlay predicted keypoints on raw videos to generate labeled MP4 videos.

In [ ]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/raw_videos"           # Input: session video folders
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"       # Input: pose predictions
DRIVE_LABELED_VIDEOS = f"{DRIVE_ROOT}/pose_outputs/labeled_videos"  # Output

CONFIDENCE_THRESHOLD = 0.5     # Skip keypoints below this confidence
KEYPOINT_RADIUS = 4            # Circle radius in pixels
LINE_THICKNESS = 2

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet opencv-python pandas numpy pyarrow

In [ ]:
from src.io.video_inventory import scan_videos

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos to visualize")
df[["filename", "session", "camera", "frame_count", "duration_min"]]

In [ ]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

# Skeleton edges (index pairs into KEYPOINT_NAMES)
SKELETON = [
    (0, 1),  # snout -> left_ear
    (0, 2),  # snout -> right_ear
    (1, 3),  # left_ear -> neck
    (2, 3),  # right_ear -> neck
    (3, 4),  # neck -> shoulders
    (4, 5),  # shoulders -> mid_back
    (5, 6),  # mid_back -> hip
    (6, 7),  # hip -> tail_base
]

# Distinct BGR colors, one per keypoint
COLORS = [
    (0, 255, 0),    # snout - green
    (255, 0, 0),    # left_ear - blue
    (0, 0, 255),    # right_ear - red
    (255, 255, 0),  # neck - cyan
    (255, 0, 255),  # shoulders - magenta
    (0, 255, 255),  # mid_back - yellow
    (128, 128, 255),# hip - pink
    (255, 128, 128),# tail_base - light blue
]

out_root = Path(DRIVE_LABELED_VIDEOS)
out_root.mkdir(parents=True, exist_ok=True)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Rendering labeled videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    session = row["session"]
    camera = row["camera"]
    stem = Path(row["filename"]).stem
    fps = row["fps"]
    width = int(row["width"])
    height = int(row["height"])

    # Corresponding pose parquet
    pose_file = Path(DRIVE_POSE_OUTPUTS) / session / f"{stem}_cam{camera}_pose.parquet"
    if not pose_file.exists():
        print(f"  Skipping {stem} — no pose file at {pose_file}")
        continue

    out_file = out_root / session / f"{stem}_cam{camera}_labeled.mp4"
    if out_file.exists():
        print(f"  Skipping {out_file.name} (already exists)")
        continue

    # Load predicted keypoints
    pose_df = pd.read_parquet(str(pose_file))

    # Open source video
    cap = cv2.VideoCapture(str(video_path))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_file), fourcc, fps, (width, height))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx < len(pose_df):
            row_data = pose_df.iloc[frame_idx]
            # Collect visible keypoints for this frame
            pts = {}
            for i, kp in enumerate(KEYPOINT_NAMES):
                cx = row_data[f"{kp}_x"]
                cy = row_data[f"{kp}_y"]
                conf = row_data[f"{kp}_likelihood"]
                if conf >= CONFIDENCE_THRESHOLD:
                    pts[i] = (int(round(cx)), int(round(cy)))

            # Draw skeleton edges
            for i, j in SKELETON:
                if i in pts and j in pts:
                    cv2.line(frame, pts[i], pts[j], COLORS[i], LINE_THICKNESS)

            # Draw keypoint circles
            for i, pt in pts.items():
                cv2.circle(frame, pt, KEYPOINT_RADIUS, COLORS[i], -1)
                cv2.circle(frame, pt, KEYPOINT_RADIUS, (255, 255, 255), 1)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"  Saved: {out_file} ({frame_idx} frames)")

print("\nAll labeled videos generated!")

In [ ]:
# Verify outputs
out_root = Path(DRIVE_LABELED_VIDEOS)
output_files = list(out_root.rglob("*.mp4"))
print(f"Total labeled videos: {len(output_files)}")
for f in output_files:
    print(f"  {f.relative_to(out_root)}")